# 3x Leveraged ETF — DSL Engine & Comparison

Express the 50/50 TQQQ/TMF strategy in QuantDSL, compare with manual implementation.

**Parent notebook:** `triple_leveraged_etf_strategy.ipynb` (Part 1: Manual Strategy)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from pathlib import Path
from datetime import datetime, timedelta
import sys, warnings
warnings.filterwarnings('ignore')

import subprocess as _sp
PROJECT_ROOT = Path(_sp.check_output(
    "git rev-parse --show-toplevel", shell=True, text=True, cwd="."
).strip())

sys.path.insert(0, str(PROJECT_ROOT / 'signum'))
from signum import Chart, Dashboard

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / 'equities' / 'triple_leveraged_etfs.parquet')
prices = df.pivot(index='date', columns='ticker', values='close').sort_index().dropna()

returns = prices.pct_change().dropna()

In [ ]:
# Strategy constants (must match manual notebook)
REBALANCE_MONTHS = 2
CRASH_THRESHOLD = -0.20
TARGET_WEIGHT_TQQQ = 0.50
TARGET_WEIGHT_TMF = 0.50
INITIAL_CAPITAL = 100_000
SLIPPAGE_PCT = 0.0025

def calculate_metrics(portfolio_values: pd.Series, risk_free_rate: float = 0.02) -> dict:
    """Calculate key performance metrics for a portfolio equity curve."""
    daily_returns = portfolio_values.pct_change().dropna()
    years = (portfolio_values.index[-1] - portfolio_values.index[0]).days / 365.25
    total_return = portfolio_values.iloc[-1] / portfolio_values.iloc[0] - 1
    cagr = (1 + total_return) ** (1/years) - 1
    volatility = daily_returns.std() * np.sqrt(252)
    sharpe = (cagr - risk_free_rate) / volatility if volatility > 0 else 0
    downside_std = daily_returns[daily_returns < 0].std() * np.sqrt(252)
    sortino = (cagr - risk_free_rate) / downside_std if downside_std > 0 else 0
    drawdown = (portfolio_values - portfolio_values.cummax()) / portfolio_values.cummax()
    max_drawdown = drawdown.min()
    mar_ratio = abs(cagr / max_drawdown) if max_drawdown != 0 else float('inf')
    return {
        'Years': years, 'Total Return': total_return * 100, 'CAGR': cagr * 100,
        'Volatility': volatility * 100, 'Sharpe Ratio': sharpe, 'Sortino Ratio': sortino,
        'Max Drawdown': max_drawdown * 100, 'MAR Ratio': mar_ratio,
    }


# Strategy Parameters
REBALANCE_MONTHS = 2
CRASH_THRESHOLD = -0.20
TARGET_WEIGHT_TQQQ = 0.50
TARGET_WEIGHT_TMF = 0.50
INITIAL_CAPITAL = 100_000
SLIPPAGE_PCT = 0.0025

def run_strategy(prices: pd.DataFrame, returns: pd.DataFrame) -> pd.DataFrame:
    """Run the 3x Leveraged ETF strategy with bimonthly rebalancing and crash filter."""
    dates = prices.index
    portfolio_values = []
    in_cash = False
    pre_crash_tqqq_price = None
    last_rebalance_month = None
    capital = INITIAL_CAPITAL
    shares_tqqq = shares_tmf = 0.0

    for i, date in enumerate(dates):
        tqqq_price = prices.loc[date, 'TQQQ']
        tmf_price = prices.loc[date, 'TMF']

        if i > 0 and not in_cash:
            capital = shares_tqqq * tqqq_price + shares_tmf * tmf_price

        # Crash filter
        if i > 0 and not in_cash:
            tqqq_ret = returns.loc[date, 'TQQQ'] if date in returns.index else 0
            if tqqq_ret <= CRASH_THRESHOLD:
                in_cash = True
                pre_crash_tqqq_price = prices.loc[dates[i-1], 'TQQQ']
                capital *= (1 - SLIPPAGE_PCT)
                shares_tqqq = shares_tmf = 0.0

        # Re-entry
        if in_cash and pre_crash_tqqq_price is not None and tqqq_price > pre_crash_tqqq_price:
            in_cash = False
            pre_crash_tqqq_price = None
            capital *= (1 - SLIPPAGE_PCT)
            shares_tqqq = (capital * TARGET_WEIGHT_TQQQ) / tqqq_price
            shares_tmf = (capital * TARGET_WEIGHT_TMF) / tmf_price
            last_rebalance_month = (date.year, date.month)

        # First day init
        if i == 0 and not in_cash:
            shares_tqqq = (capital * TARGET_WEIGHT_TQQQ) / tqqq_price
            shares_tmf = (capital * TARGET_WEIGHT_TMF) / tmf_price
            last_rebalance_month = (date.year, date.month)

        # Bimonthly rebalance
        current_month = (date.year, date.month)
        if not in_cash and last_rebalance_month is not None:
            months_diff = (current_month[0] - last_rebalance_month[0]) * 12 + \
                          (current_month[1] - last_rebalance_month[1])
            if months_diff >= REBALANCE_MONTHS:
                total_value = shares_tqqq * tqqq_price + shares_tmf * tmf_price
                rebal_cost = abs(shares_tqqq * tqqq_price - total_value * TARGET_WEIGHT_TQQQ) * SLIPPAGE_PCT
                total_value -= rebal_cost
                shares_tqqq = (total_value * TARGET_WEIGHT_TQQQ) / tqqq_price
                shares_tmf = (total_value * TARGET_WEIGHT_TMF) / tmf_price
                capital = total_value
                last_rebalance_month = current_month

        if in_cash:
            port_value = capital
            w_tqqq, w_tmf = 0.0, 0.0
        else:
            tqqq_value = shares_tqqq * tqqq_price
            tmf_value = shares_tmf * tmf_price
            port_value = tqqq_value + tmf_value
            w_tqqq = tqqq_value / port_value if port_value > 0 else 0
            w_tmf = tmf_value / port_value if port_value > 0 else 0

        portfolio_values.append({
            'date': date, 'portfolio_value': port_value,
            'weight_tqqq': w_tqqq, 'weight_tmf': w_tmf,
            'in_cash': in_cash, 'shares_tqqq': shares_tqqq, 'shares_tmf': shares_tmf,
        })

    return pd.DataFrame(portfolio_values).set_index('date')

strategy_results = run_strategy(prices, returns)
print(f"${INITIAL_CAPITAL:,} â†’ ${strategy_results['portfolio_value'].iloc[-1]:,.0f}  |  "
      f"{strategy_results.index.min()} â†’ {strategy_results.index.max()}")

print(f'Manual: ${INITIAL_CAPITAL:,} \u2192 ${strategy_results["portfolio_value"].iloc[-1]:,.0f}')


---
## Part 2: DSL Engine Implementation
Express the strategy in QuantDSL and run through the backtest engine. **Limitation:** the crash filter (stateful regime switching) can't be expressed in current DSL signals — this tests the core 50/50 daily rebalancing only.

In [ ]:
import sys, os
import subprocess as _sp
project_root = _sp.check_output(
    "git rev-parse --show-toplevel", shell=True, text=True, cwd="."
).strip()
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.chdir(project_root)

In [ ]:
from quantdsl_backtest.dsl.strategy import Strategy
from quantdsl_backtest.dsl.data_config import DataConfig
from quantdsl_backtest.dsl.universe import Universe, HasHistory
from quantdsl_backtest.dsl.factors import ReturnFactor, VolatilityFactor
from quantdsl_backtest.dsl.signals import (
    CrossSectionRank, NotNull, And, MaskFromBoolean, LessEqual, Quantile,
)
from quantdsl_backtest.dsl.portfolio import (
    LongShortPortfolio, Book, TopN, BottomN, EqualWeight,
)
from quantdsl_backtest.dsl.execution import (
    Execution, OrderPolicy, LatencyModel, PowerLawSlippageModel, VolumeParticipation,
)
from quantdsl_backtest.dsl.costs import Costs, Commission, BorrowCost, FinancingCost, StaticFees
from quantdsl_backtest.dsl.backtest_config import BacktestConfig, Reporting, RiskChecks
from quantdsl_backtest.engine.analytics.types import StrategyAnalyticsConfig


def build_triple_leveraged_strategy() -> Strategy:
    """3x Leveraged ETF 50/50 expressed in QuantDSL â€” daily rebalance, no crash filter."""

    data_cfg = DataConfig(
        source="parquet://equities/triple_leveraged_etfs.parquet",
        calendar="XNYS", frequency="1d",
        start="2010-02-11", end="2026-03-03",
        price_adjustment="split_dividend",
        fields=["open", "high", "low", "close", "volume"],
    )

    universe = Universe(
        name="TripleLeveragedETFs", id_field="ticker",
        static_instruments=["TQQQ", "TMF"],
        filters=[HasHistory(min_days=5)],
    )

    factors = {
        "daily_ret": ReturnFactor(name="daily_ret", field="close", lookback=1, method="simple"),
        "vol_20d": VolatilityFactor(name="vol_20d", field="close", lookback=20, method="realized"),
    }

    eligible = NotNull("daily_ret"); eligible.name = "eligible"
    rank = CrossSectionRank(factor_name="daily_ret", mask_name="eligible", method="percentile", name="rank")
    long_candidates = MaskFromBoolean(name="long_candidates", expr=NotNull("daily_ret"))
    short_candidates = MaskFromBoolean(name="short_candidates",
        expr=LessEqual(left="rank", right=-999.0))

    signals = {"eligible": eligible, "rank": rank,
               "long_candidates": long_candidates, "short_candidates": short_candidates}

    # TopN(n=2) + BottomN(n=0) â†’ both instruments selected, long-only
    portfolio = LongShortPortfolio(
        long_book=Book(name="long_book",
            selector=TopN(factor_name="rank", n=2, mask_name="long_candidates"),
            weighting=EqualWeight()),
        short_book=Book(name="short_book",
            selector=BottomN(factor_name="rank", n=0, mask_name="short_candidates"),
            weighting=EqualWeight()),
        rebalance_frequency="1d", rebalance_at="market_close", signal_delay_bars=0,
        target_gross_leverage=1.0, target_net_exposure=1.0,
        max_abs_weight_per_name=0.55,
    )

    execution = Execution(
        order_policy=OrderPolicy(default_order_type="MOC", time_in_force="DAY"),
        latency=LatencyModel(signal_to_order_delay_bars=0, market_latency_ms=0),
        slippage=PowerLawSlippageModel(base_bps=2.5, k=10.0, exponent=0.5),
        volume_limits=VolumeParticipation(max_participation=1.0, mode="proportional"),
    )

    costs = Costs(
        commission=Commission(type="bps_notional", amount=1.0),
        borrow=BorrowCost(default_annual_rate=0.0),
        financing=FinancingCost(base_rate_curve="SOFR", spread_bps=0),
        fees=StaticFees(nav_fee_annual=0.0),
    )

    backtest_cfg = BacktestConfig(
        engine="event_driven", cash_initial=100_000.0,
        risk_checks=RiskChecks(),
        reporting=Reporting(strategyAnalytics=StrategyAnalyticsConfig(
            output_dir="outputs/triple_leveraged_etf",
            title="3x Leveraged ETF 50/50 (QuantDSL)")),
    )

    return Strategy(
        name="triple_leveraged_etf_5050", data=data_cfg, universe=universe,
        factors=factors, signals=signals, portfolio=portfolio,
        execution=execution, costs=costs, backtest=backtest_cfg,
    )

strategy = build_triple_leveraged_strategy()
print(f"Strategy: {strategy.name}  |  {strategy.universe.static_instruments}  |  engine={strategy.backtest.engine}")

In [ ]:
from quantdsl_backtest.engine.backtest_runner import run_backtest

result = run_backtest(strategy)
print(f"DSL: ${result.equity.iloc[0]:,.0f} â†’ ${result.equity.iloc[-1]:,.0f}  |  "
      f"{result.start_date} â†’ {result.end_date}  |  {len(result.equity)} bars  |  "
      f"Sharpe {result.metrics.get('sharpe_ratio', 0):.3f}")

### DSL Positions, Weights & Trades

In [ ]:
# ====== DSL Engine Weights & Equity — Signum ======
Dashboard(
    panes=[
        (Chart(theme='midnight', height=150)
         .line(result.weights['TQQQ'] * 100, name='TQQQ wt%', color='#2196F3', width=1)
         .line(result.weights['TMF'] * 100, name='TMF wt%', color='#4CAF50', width=1)
         .price_line(50, title='Target 50%', color='gray')),
        (Chart(theme='midnight', height=250)
         .line(result.equity, name='DSL Equity', color='#9C27B0', width=2)),
    ],
    titles=['DSL Engine: Portfolio Weights (%)', 'DSL Engine: Equity ($)'],
    theme='midnight',
).show()

In [ ]:
trades = result.trades.copy()

# Cost breakdown
total_commission = trades['commission'].sum() if 'commission' in trades.columns else 0
total_fees = trades['fees'].sum() if 'fees' in trades.columns else 0
if 'slippage_bps' in trades.columns and 'notional' in trades.columns:
    trades['slippage_dollars'] = trades['notional'].abs() * trades['slippage_bps'] / 10_000
    total_slippage = trades['slippage_dollars'].sum()
else:
    total_slippage = trades.get('slippage', pd.Series([0])).sum()

total_costs = total_slippage + total_commission + total_fees
total_notional = trades['notional'].abs().sum()

cost_summary = pd.DataFrame({
    'Amount': [total_slippage, total_commission, total_fees, total_costs, total_notional],
    'Detail': [
        f'{total_slippage/result.equity.iloc[-1]*100:.2f}% of final equity',
        f'{total_commission/result.equity.iloc[-1]*100:.2f}% of final equity',
        '', f'{total_costs/total_notional*10000:.1f} bps avg cost per $ traded',
        f'~${total_costs/16:,.0f}/yr cost drag',
    ]
}, index=['Slippage', 'Commission', 'Fees', 'Total Costs', 'Total Notional'])
print(f"{len(trades)} trades  |  All-in costs: ${total_costs:,.0f} ({total_costs/result.equity.iloc[-1]*100:.1f}% of final equity)")

display_cols = [c for c in ['datetime', 'instrument', 'side', 'quantity', 'price',
                            'notional', 'slippage_bps', 'slippage_dollars', 'commission']
                if c in trades.columns]
trades[display_cols].head(10)

In [ ]:
from quantdsl_backtest.engine.results import BacktestResult

weights = result.weights.copy()
if 'datetime' in trades.columns:
    daily_orders = trades.groupby(['datetime', 'instrument'])['quantity'].sum().unstack(fill_value=0)
    daily_notional = trades.groupby('datetime')['notional'].apply(lambda x: x.abs().sum())
else:
    daily_orders = trades.groupby([trades.index, 'instrument'])['quantity'].sum().unstack(fill_value=0)
    daily_notional = trades.groupby(trades.index)['notional'].apply(lambda x: x.abs().sum())

weekly_orders = daily_orders.resample('W').sum()
weekly_notional = daily_notional.resample('W').sum()
weekly_equity = result.equity.resample('W').last()
weekly_turnover_pct = (weekly_notional / weekly_equity * 100).dropna()

Dashboard(
    panes=[
        (Chart(theme='midnight', height=160)
         .line(weights['TQQQ'] * 100, name='TQQQ wt%', color='#2196F3', width=1)
         .line(weights['TMF'] * 100, name='TMF wt%', color='#4CAF50', width=1)
         .price_line(50, title='Target', color='gray')),
        (Chart(theme='midnight', height=130)
         .histogram(weekly_orders.get('TQQQ', pd.Series(dtype=float)),
                    name='TQQQ orders', color='#2196F3')
         .histogram(weekly_orders.get('TMF', pd.Series(dtype=float)),
                    name='TMF orders', color='#4CAF50')),
        (Chart(theme='midnight', height=110)
         .histogram(weekly_turnover_pct, name='Turnover %NAV', color='steelblue')),
    ],
    titles=['Portfolio Weights (%)', 'Weekly Order Flow (shares)', 'Weekly Turnover (% NAV)'],
    theme='midnight',
).show()

print(f"Weekly turnover: median {weekly_turnover_pct.median():.1f}% NAV  |  max {weekly_turnover_pct.max():.0f}% NAV")

---
## Part 3: Manual vs DSL Comparison

In [ ]:
# Build combined Manual vs DSL comparison DataFrame
manual_eq = strategy_results['portfolio_value'].rename('Manual')

dsl_eq = result.equity.copy()
dsl_eq.name = 'DSL'

combined = pd.concat([manual_eq, dsl_eq], axis=1).dropna()
print(f"Overlap: {len(combined)} days  |  {combined.index.min()} to {combined.index.max()}")

# Metrics comparison
manual_m = calculate_metrics(combined['Manual'])
dsl_m = calculate_metrics(combined['DSL'])
pd.DataFrame({'Manual': manual_m, 'DSL Engine': dsl_m})

### DSL vs Manual: Why the ~$700K Difference?

The DSL engine returns **$4.2M** vs manual's **$3.5M**. This isn't a cost mismatch â€” it's because the strategies differ:

| Feature | Manual | DSL Engine |
|---|---|---|
| Rebalancing | Every 2 months | Daily |
| Crash filter | Yes (exit on -20% TQQQ day) | No (DSL can't express stateful regime) |
| Slippage | 0.25% per trade | 2.5 bps power-law model |
| Commission | None | 1 bps per trade |
| Cost frequency | ~6 trades/year | Daily micro-rebalance |

**Why DSL outperforms despite more frequent costs**: Daily rebalancing systematically harvests the "rebalancing premium" by selling the daily winner and buying the loser. Over 16 years this dominates. The crash filter in manual also causes it to miss part of the recovery rally after COVID.

In [ ]:
# Equity comparison: Manual vs DSL — full period + COVID zoom
_cash_period = strategy_results[strategy_results['in_cash']].index
_cash_pos = pd.Series(0, index=combined.index, name='position')
_cash_pos.loc[_cash_pos.index.isin(_cash_period)] = 1
_cash_df = _cash_pos.to_frame('position')

covid_slice = slice('2020-02-15', '2020-06-01')

Dashboard(
    panes=[
        (Chart(theme='midnight', height=280, watermark='Full Period')
         .line(combined['Manual'], name='Manual', color='#2196F3', width=2)
         .line(combined['DSL'], name='DSL', color='#9C27B0', width=2)
         .shade(_cash_df, position_col='position', color='#FF9800', opacity=0.12)),
        (Chart(theme='midnight', height=200, watermark='COVID Zoom (Feb–Jun 2020)')
         .line(combined.loc[covid_slice, 'Manual'], name='Manual', color='#2196F3', width=2)
         .line(combined.loc[covid_slice, 'DSL'], name='DSL', color='#9C27B0', width=2)
         .shade(_cash_df.loc[covid_slice], position_col='position', color='#FF9800', opacity=0.15)),
    ],
    titles=['Equity: Manual vs DSL (orange = manual in cash)', 'COVID Crash Zoom'],
    theme='midnight',
).show()

In [ ]:
# Equity Difference Over Time
equity_diff = combined['DSL'] - combined['Manual']

man_w = strategy_results['weight_tqqq'].reindex(combined.index).ffill() * 100
dsl_w = result.weights['TQQQ'].reindex(combined.index).ffill() * 100

Dashboard(
    panes=[
        (Chart(theme='midnight', height=220)
         .line(combined['Manual'], name='Manual (bimonthly + crash filter)', color='#2196F3', width=2)
         .line(combined['DSL'], name='DSL (daily rebal)', color='#9C27B0', width=2)),
        (Chart(theme='midnight', height=140)
         .baseline(equity_diff.dropna(), base_value=0)),
        (Chart(theme='midnight', height=100)
         .line(man_w, name='Manual TQQQ wt%', color='#2196F3', width=1)
         .line(dsl_w, name='DSL TQQQ wt%', color='#9C27B0', width=1)
         .price_line(50, title='Target', color='gray')),
    ],
    titles=['Daily Equity: Manual vs DSL', 'Equity Difference (DSL − Manual)',
            'TQQQ Weight % Comparison'],
    theme='midnight',
).show()

In [ ]:
# Positions & weights side-by-side comparison (Manual vs DSL)
dsl_weights = result.weights
dsl_tqqq_w = dsl_weights.iloc[:, 0] * 100 if len(dsl_weights.columns) >= 2 else pd.Series(dtype=float)

Dashboard(
    panes=[
        (Chart(theme='midnight', height=160)
         .line(strategy_results['shares_tqqq'], name='Manual TQQQ', color='#9C27B0', width=1)),
        (Chart(theme='midnight', height=160)
         .line(strategy_results['shares_tmf'], name='Manual TMF', color='#9C27B0', width=1)),
        (Chart(theme='midnight', height=120)
         .line(strategy_results['weight_tqqq'] * 100, name='Manual wt%', color='#9C27B0', width=1)
         .line(dsl_tqqq_w, name='DSL wt%', color='#FF9800', width=1)
         .price_line(50, title='Target', color='gray')),
    ],
    titles=['TQQQ Shares (Manual)', 'TMF Shares (Manual)', 'TQQQ Weight % (Manual vs DSL)'],
    theme='midnight',
).show()

### Implementation Differences

In [ ]:
differences = pd.DataFrame({
    'Feature': [
        'Rebalancing', 'Crash filter', 'Re-entry logic', 'Slippage model',
        'Commission', 'Position sizing', 'Execution', 'Daily weights',
        'Stateful logic', 'Output: trades', 'Lines of code',
    ],
    'Python (Manual)': [
        'Every 2 months', 'Exit on -20% TQQQ day', 'Re-enter above pre-crash price',
        'Flat 0.25%', 'None', 'Dollar-based', 'Fill at close',
        'Drifts between rebalances', 'Fully supported (in_cash flag)', 'Not tracked', '~130',
    ],
    'DSL Engine': [
        'Daily', 'Not supported (stateless signals)', 'Always invested',
        'PowerLaw 2.5 bps + volume', '1 bps/trade', 'Weight-based', 'MOC + slippage',
        'Maintained at 50/50 daily', 'Not supported', f'{len(trades)} trades logged', '~180',
    ],
})

diff_val = combined['DSL'].iloc[-1] - combined['Manual'].iloc[-1]
diff_pct = diff_val / combined['Manual'].iloc[-1] * 100
print(f"Final equity — Manual: ${combined['Manual'].iloc[-1]:,.0f}  |  "
      f"DSL: ${combined['DSL'].iloc[-1]:,.0f}  |  "
      f"Diff: ${diff_val:,.0f} ({diff_pct:.1f}%)")
differences.set_index('Feature')

## Current Positions & Parameter Optimization

In [ ]:
today = strategy_results.index.max()
last = strategy_results.iloc[-1]
port_val = last['portfolio_value']
in_cash = last['in_cash']
tqqq_px = prices.loc[today, 'TQQQ']
tmf_px = prices.loc[today, 'TMF']
sh_tqqq, sh_tmf = last['shares_tqqq'], last['shares_tmf']

snapshot = pd.DataFrame({
    'Shares': [sh_tqqq, sh_tmf, ''],
    'Price': [f'${tqqq_px:.2f}', f'${tmf_px:.2f}', ''],
    'Value': [f'${sh_tqqq*tqqq_px:,.0f}', f'${sh_tmf*tmf_px:,.0f}', f'${port_val:,.0f}'],
    'Weight': [f'{last["weight_tqqq"]:.1%}', f'{last["weight_tmf"]:.1%}', '100%'],
}, index=['TQQQ', 'TMF', 'Total'])

status = 'IN CASH (crash filter)' if in_cash else 'FULLY INVESTED'
print(f"Portfolio Snapshot â€” {today.strftime('%Y-%m-%d')}  |  {status}  |  ${port_val:,.0f}")

# Drift from target
if not in_cash:
    print(f"Weight drift from 50/50: TQQQ {last['weight_tqqq']-0.5:+.1%}  |  TMF {last['weight_tmf']-0.5:+.1%}")

# To replicate with $100K
print(f"\nTo replicate with $100K: Buy {50000/tqqq_px:,.0f} TQQQ + {50000/tmf_px:,.0f} TMF, rebalance every {REBALANCE_MONTHS}mo")
snapshot

In [ ]:
from itertools import product

def run_strategy_fast(prices, returns, rebal_months, crash_thresh, w_tqqq_target, slippage=0.0025):
    """Fast strategy runner for parameter sweeps."""
    dates = prices.index
    capital = 100_000.0
    shares_tqqq = (capital * w_tqqq_target) / prices.iloc[0]['TQQQ']
    shares_tmf  = (capital * (1 - w_tqqq_target)) / prices.iloc[0]['TMF']
    in_cash = False; pre_crash_px = None
    last_rebal = (dates[0].year, dates[0].month)
    equity = np.empty(len(dates))

    for i in range(len(dates)):
        dt = dates[i]
        px_t, px_m = prices.iloc[i]['TQQQ'], prices.iloc[i]['TMF']
        if i > 0 and not in_cash:
            capital = shares_tqqq * px_t + shares_tmf * px_m
        if i > 0 and not in_cash and dt in returns.index:
            if returns.loc[dt, 'TQQQ'] <= crash_thresh:
                in_cash = True; pre_crash_px = prices.iloc[i-1]['TQQQ']
                capital *= (1 - slippage); shares_tqqq = shares_tmf = 0.0
        if in_cash and pre_crash_px is not None and px_t > pre_crash_px:
            in_cash = False; pre_crash_px = None; capital *= (1 - slippage)
            shares_tqqq = (capital * w_tqqq_target) / px_t
            shares_tmf  = (capital * (1 - w_tqqq_target)) / px_m
            last_rebal = (dt.year, dt.month)
        cm = (dt.year, dt.month)
        if not in_cash and last_rebal is not None:
            md = (cm[0] - last_rebal[0]) * 12 + (cm[1] - last_rebal[1])
            if md >= rebal_months:
                tv = shares_tqqq * px_t + shares_tmf * px_m
                tv -= abs(shares_tqqq * px_t - tv * w_tqqq_target) * slippage
                shares_tqqq = (tv * w_tqqq_target) / px_t
                shares_tmf  = (tv * (1 - w_tqqq_target)) / px_m
                capital = tv; last_rebal = cm
        equity[i] = capital if in_cash else (shares_tqqq * px_t + shares_tmf * px_m)

    daily_rets = np.diff(equity) / equity[:-1]
    ann_ret = (equity[-1] / equity[0]) ** (252.0 / len(equity)) - 1
    ann_vol = np.std(daily_rets) * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    sortino_vol = np.std(daily_rets[daily_rets < 0]) * np.sqrt(252) if np.any(daily_rets < 0) else 1
    max_dd = np.min(equity / np.maximum.accumulate(equity) - 1)
    return sharpe, ann_ret / sortino_vol, ann_ret / abs(max_dd) if max_dd != 0 else 0, ann_ret, ann_vol, max_dd, equity[-1]

# Parameter grid
rebal_months_grid = [1, 2, 3, 4, 6, 12]
crash_thresh_grid = [-0.10, -0.15, -0.20, -0.25, -999]
weight_grid = [0.30, 0.40, 0.50, 0.60, 0.70]

results = []
for rm, ct, wt in product(rebal_months_grid, crash_thresh_grid, weight_grid):
    s, so, ca, ar, av, md, fv = run_strategy_fast(prices, returns, rm, ct, wt)
    results.append({'Rebal (mo)': rm, 'Crash Thr': f'{ct:.0%}' if ct > -100 else 'None',
                    'TQQQ Wt': f'{wt:.0%}', 'Sharpe': s, 'Sortino': so, 'Calmar': ca,
                    'Ann Ret': ar, 'Ann Vol': av, 'Max DD': md, 'Final ($K)': fv/1000})

grid_df = pd.DataFrame(results).sort_values('Sharpe', ascending=False)

# Current vs best
curr = grid_df[(grid_df['Rebal (mo)'] == REBALANCE_MONTHS) &
               (grid_df['Crash Thr'] == f'{CRASH_THRESHOLD:.0%}') &
               (grid_df['TQQQ Wt'] == f'{TARGET_WEIGHT_TQQQ:.0%}')].iloc[0]
best = grid_df.iloc[0]
print(f"{len(results)} combos tested  |  Current: Sharpe {curr['Sharpe']:.3f}  |  "
      f"Best: Sharpe {best['Sharpe']:.3f} ({best['Rebal (mo)']}mo / {best['Crash Thr']} / {best['TQQQ Wt']})")

top20 = grid_df.head(20).copy()
for c, fmt in [('Ann Ret', '{:.1%}'), ('Ann Vol', '{:.1%}'), ('Max DD', '{:.1%}'),
               ('Sharpe', '{:.3f}'), ('Sortino', '{:.3f}'), ('Calmar', '{:.3f}'), ('Final ($K)', '{:,.0f}')]:
    top20[c] = top20[c].map(fmt.format)
top20.reset_index(drop=True)

In [ ]:
# Sharpe heatmap: Rebalance freq x TQQQ weight (crash=-20%)
pivot_data = grid_df[grid_df['Crash Thr'] == '-20%'].pivot_table(
    index='TQQQ Wt', columns='Rebal (mo)', values='Sharpe')
pivot_dd = grid_df[grid_df['Crash Thr'] == '-20%'].pivot_table(
    index='TQQQ Wt', columns='Rebal (mo)', values='Max DD')

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Sharpe: Rebal Freq x TQQQ Weight', 'Max DD: Rebal Freq x TQQQ Weight'), horizontal_spacing=0.12)

fig.add_trace(go.Heatmap(z=pivot_data.values, x=[str(c) for c in pivot_data.columns],
    y=pivot_data.index.tolist(), colorscale='RdYlGn', text=np.round(pivot_data.values, 3),
    texttemplate='%{text:.3f}', colorbar=dict(title='Sharpe', x=0.45)), row=1, col=1)
fig.add_trace(go.Heatmap(z=pivot_dd.values*100, x=[str(c) for c in pivot_dd.columns],
    y=pivot_dd.index.tolist(), colorscale='RdYlGn', text=np.round(pivot_dd.values*100, 1),
    texttemplate='%{text:.1f}%', colorbar=dict(title='Max DD %', x=1.0)), row=1, col=2)
fig.update_xaxes(title_text='Rebalance (months)', row=1, col=1)
fig.update_xaxes(title_text='Rebalance (months)', row=1, col=2)
fig.update_yaxes(title_text='TQQQ Weight', row=1, col=1)
fig.update_layout(height=400, template='plotly_white', title='Parameter Sensitivity Heatmaps (crash=-20%)')
fig.show()

# Quick test: Inverse-Volatility Weights
lookback = 60
vol_tqqq = returns['TQQQ'].rolling(lookback).std() * np.sqrt(252)
vol_tmf  = returns['TMF'].rolling(lookback).std() * np.sqrt(252)
dyn_w_tqqq = ((1/vol_tqqq) / (1/vol_tqqq + 1/vol_tmf)).dropna()

iv_rets = (dyn_w_tqqq.shift(1) * returns['TQQQ'] + (1 - dyn_w_tqqq.shift(1)) * returns['TMF']).dropna()
static_rets = (0.5 * returns['TQQQ'] + 0.5 * returns['TMF']).loc[iv_rets.index]

# Quick test: 200-day SMA Trend Filter
sma200 = prices['TQQQ'].rolling(200).mean()
above_sma = (prices['TQQQ'] > sma200).reindex(returns.index).fillna(False)
trend_rets = returns.copy()
trend_rets.loc[~above_sma, 'TQQQ'] = 0
trend_port = (0.5 * trend_rets['TQQQ'] + 0.5 * trend_rets['TMF']).dropna()
raw_port = (0.5 * returns['TQQQ'] + 0.5 * returns['TMF']).dropna()

# Summary table
def quick_stats(r):
    s = r.mean() / r.std() * np.sqrt(252)
    md = ((1+r).cumprod() / (1+r).cumprod().cummax() - 1).min()
    ar = (1+r).prod() ** (252/len(r)) - 1
    return {'Sharpe': s, 'Max DD': md, 'Ann Return': ar}

enhancements = pd.DataFrame({
    'Static 50/50': quick_stats(static_rets),
    'Inv-Vol Weights': quick_stats(iv_rets),
    'SMA-200 Filter': quick_stats(trend_port),
}).T
enhancements['Sharpe Delta'] = enhancements['Sharpe'] - enhancements.loc['Static 50/50', 'Sharpe']

print(f"Current TQQQ ${prices['TQQQ'].iloc[-1]:.2f} vs SMA200 ${sma200.iloc[-1]:.2f} â†’ "
      f"{'HOLD' if prices['TQQQ'].iloc[-1] > sma200.iloc[-1] else 'EXIT'}")
enhancements

---
## Summary

| Approach | Final Equity | CAGR | Sharpe | Max DD |
|---|---|---|---|---|
| Manual (bimonthly + crash filter) | ~$3.5M | ~25% | ~0.78 | -77% |
| DSL Engine (daily rebal) | ~$4.2M | ~26% | ~0.87 | -75% |

**Key findings:**
- Daily rebalancing captures the "rebalancing premium" (DSL outperforms by ~$700K)
- Crash filter reduces worst drawdown but misses recovery rally
- SMA-200 trend filter offers best Sharpe improvement (+0.3) but isn't in the original strategy
- DSL can't express stateful crash filter â€” would need a `RegimeSignal` node extension